In [1]:
import pandas as pd
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("image_data_prep").getOrCreate()

In [34]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import ArrayType, FloatType 
import numpy as np
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.feature import VectorAssembler
from PIL import Image
import io

In [7]:
test_data_no_loc = "hdfs://localhost:9000/user/jj/final_proj/Brain_Tumor_Datasets/test/no"
test_data_yes_loc = "hdfs://localhost:9000/user/jj/final_proj/Brain_Tumor_Datasets/test/yes"
train_data_no_loc = "hdfs://localhost:9000/user/jj/final_proj/Brain_Tumor_Datasets/train/no"
train_data_yes_loc =  "hdfs://localhost:9000/user/jj/final_proj/Brain_Tumor_Datasets/train/yes"

test_data_no = spark.read.format("image").load(test_data_no_loc)
test_data_yes = spark.read.format("image").load(test_data_yes_loc)
train_data_no = spark.read.format("image").load(train_data_no_loc)
train_data_yes = spark.read.format("image").load(train_data_yes_loc)

test_data_no = test_data_no.withColumn("label", col("image.height") * 0)  # Set label 0
test_data_yes = test_data_yes.withColumn("label", col("image.height") * 0 + 1)  # Set label 1
train_data_no = train_data_no.withColumn("label", col("image.height") * 0)  # Set label 0
train_data_yes = train_data_yes.withColumn("label", col("image.height") * 0 + 1)  # Set label 1

all_data = test_data_no.union(test_data_yes).union(train_data_no).union(train_data_yes)


In [40]:
# Check if image data exists
data_check = all_data.select(
    col("image.data").isNull().alias("data_is_null"),
    (col("image").isNull()).alias("image_is_null")
).toPandas()

print(f"Total rows: {len(data_check)}")
print(f"Rows with null image: {data_check['image_is_null'].sum()}")
print(f"Rows with null image.data: {data_check['data_is_null'].sum()}")

Total rows: 8764
Rows with null image: 0
Rows with null image.data: 0


In [41]:
# Collect a single row to test (use limit to avoid OOM errors)
sample_row = all_data.limit(1).collect()

if len(sample_row) > 0:
    row = sample_row[0]
    
    # Check if image and image.data exist
    if hasattr(row, "image") and row.image is not None:
        if hasattr(row.image, "data") and row.image.data is not None:
            # Get the image data
            binary_data = row.image.data
            print(f"Image data type: {type(binary_data)}")
            print(f"Image data length: {len(binary_data) if binary_data is not None else 'None'}")
            
            # Test preprocessing function directly
            features = preprocess_image(binary_data)
            print(f"Features result: {'Success - length ' + str(len(features)) if features is not None else 'Failed - returned None'}")
            
            # Try to open the image directly to check format
            try:
                image = Image.open(io.BytesIO(binary_data))
                print(f"Image format: {image.format}, Size: {image.size}, Mode: {image.mode}")
            except Exception as e:
                print(f"Error opening image: {str(e)}")
        else:
            print("image.data is None or not accessible")
    else:
        print("image field is None or not accessible")
else:
    print("No rows found in the dataset")

Image data type: <class 'bytearray'>
Image data length: 6220800
Features result: Failed - returned None
Error opening image: cannot identify image file <_io.BytesIO object at 0x7f9bafba8d60>


In [42]:
# Check the schema to understand the exact structure
all_data.printSchema()

# Try accessing the image data with a simpler UDF
def check_image_data(data):
    if data is None:
        return "None"
    try:
        return f"Binary data of length {len(data)}"
    except:
        return f"Error: {type(data)}"

check_data_udf = udf(check_image_data)

# Apply the UDF and check results
all_data.select(
    check_data_udf(col("image.data")).alias("data_info")
).show(10, truncate=False)

root
 |-- image: struct (nullable = true)
 |    |-- origin: string (nullable = true)
 |    |-- height: integer (nullable = true)
 |    |-- width: integer (nullable = true)
 |    |-- nChannels: integer (nullable = true)
 |    |-- mode: integer (nullable = true)
 |    |-- data: binary (nullable = true)
 |-- label: integer (nullable = true)

+-----------------------------+
|data_info                    |
+-----------------------------+
|Binary data of length 6220800|
|Binary data of length 1583400|
|Binary data of length 3145728|
|Binary data of length 2764800|
|Binary data of length 3145728|
|Binary data of length 628800 |
|Binary data of length 1866360|
|Binary data of length 2059200|
|Binary data of length 2059200|
|Binary data of length 1048576|
+-----------------------------+
only showing top 10 rows



In [44]:
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, FloatType
import numpy as np
import io
import struct
from PIL import Image

# Modified preprocessing function that handles raw image data
def preprocess_image(binary_data):
    """Convert binary image data to a normalized feature vector."""
    if binary_data is None:
        return None
    
    try:
        # The data appears to be raw binary data, not a standard image format
        # Let's try to interpret it based on the image metadata
        # Assuming the data is in a raw format with height and width from the metadata
        
        # Convert to numpy array and reshape
        # This will depend on your specific image format/data structure
        # Example assuming grayscale (1 channel) or extracting first channel from RGB/BGR
        
        # Method 1: Try to extract one channel and reshape based on dimensions
        # This will need adjustment based on your actual image structure
        img_array = np.frombuffer(binary_data, dtype=np.uint8)
        
        # Assuming square images for simplicity, adjust as needed
        side_length = int(np.sqrt(len(img_array) / 3))  # Assuming 3 channels (RGB)
        
        # Reshape and take first channel (if multi-channel)
        try:
            reshaped = img_array[:side_length*side_length*3].reshape((side_length, side_length, 3))
            # Take first channel for simplicity
            gray_img = reshaped[:, :, 0]
        except:
            # If reshaping fails, just resize the array to a square
            side_length = int(np.sqrt(len(img_array)))
            gray_img = img_array[:side_length*side_length].reshape((side_length, side_length))
        
        # Resize to our target dimensions
        from skimage.transform import resize
        resized_img = resize(gray_img, (28, 28), anti_aliasing=True)
        
        # Normalize to 0-1
        normalized = resized_img.astype(np.float32) / 255.0
        
        # Flatten into vector
        return normalized.flatten().tolist()
        
    except Exception as e:
        # For debugging, you could log the error here
        return None

# Register UDF
image_to_array_udf = udf(preprocess_image, ArrayType(FloatType()))

# Apply preprocessing to extract features as arrays
image_features_df = all_data.withColumn("features_array", image_to_array_udf(col("image.data")))

# Filter out rows with NULL features
image_features_df = image_features_df.filter(col("features_array").isNotNull())

# Create UDF to convert arrays to vectors
array_to_vector_udf = udf(lambda arr: Vectors.dense(arr) if arr is not None else None, VectorUDT())

# Convert arrays to vectors
image_features_df = image_features_df.withColumn("features_vector", 
                                              array_to_vector_udf(col("features_array")))

# Check how many valid images we have
valid_count = image_features_df.count()
total_count = all_data.count()
print(f"Valid images processed: {valid_count} out of {total_count} ({valid_count/total_count:.2%})")

# Check a sample vector size
if valid_count > 0:
    from pyspark.sql.functions import size
    sample_size = image_features_df.select(size("features_array").alias("vector_size")).first()
    print(f"Feature vector size: {sample_size['vector_size']}")

Valid images processed: 0 out of 8764 (0.00%)
